# 第6回　CSVを読み込む・整形・記述統計
## ―― なぜ人間に読みやすい表は、機械に読ませられないのか

情報活用　／　北星学園大学　2026年度後期

今日は、**世界中の霊長類376種**を集めた本物の研究データを使う。生物学者が論文のために何十年もかけて集めた測定値を、1つの表にまとめたものである。

そして、このデータには**罠が仕掛けてある**。仕掛けたのは私ではない。データを作った研究者が、当時の慣習に従って入れたものである。

> コードはAIに書かせてよい。ただし **読んで、何をしているか説明できること**。
> 説明できないコードの出力は、あなたの根拠にならない。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

def _build_from_source(keep_missing_code=False):
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    out = out.sort_values("学名").reset_index(drop=True)
    return out if keep_missing_code else out.replace(-999, np.nan)

try:
    df = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. きれいなデータとは何か

**機械判読可能なデータ**には、4つの条件がある。

| 条件 | 意味 |
|---|---|
| **1行1件** | 1行が1件分（このデータでは1種）。1行に2件を詰めない |
| **1列1変数** | 1列が1つの項目。「体重・体長」を1列に混ぜない |
| **セルを結合しない** | 見出しをまたぐ結合は、読み込むと空欄になる |
| **色で意味を持たせない** | 赤いセル＝要注意、は機械に伝わらない。列を作る |

さらにもう1つ。**単位を混ぜない。**「10kg」と「10000g」が同じ列にあると、その列は数値ではなく文字列になり、平均が計算できなくなる。

まず、人間向けに作られた表が壊れるところを見る。

In [ ]:
# 「人間が読みやすいように」作られた表を再現する。
# ―― 見出しの上にタイトル行、結合セルのつもりの空欄、単位つきの数字、備考行。
dirty = """霊長類 体重まとめ,,,
作成: 情報活用,,,
,,,
科,種数,平均体重,備考
オナガザル科,92,7.2kg,
,,,※旧世界ザル
オマキザル科,39,796g,
コビトキツネザル科,13,,未測定
"""
with open("dirty.csv", "w", encoding="utf-8") as f:
    f.write(dirty)

bad = pd.read_csv("dirty.csv")
bad

見出しが `霊長類 体重まとめ` になってしまった。**1行目を見出しだと思い込む**のが `read_csv` の既定の動作だからだ。

見出しの位置を教えてやれば直るだろうか。

In [ ]:
bad2 = pd.read_csv("dirty.csv", skiprows=3)
print(bad2)
print()
print("平均体重の型:", bad2["平均体重"].dtype)
print()
try:
    print(bad2["平均体重"].mean())
except Exception as e:
    print("失敗:", type(e).__name__, e)

表の形にはなった。しかし **`平均体重` の型は `object`（＝文字列）** のままで、平均が出せない。

- `7.2kg` `796g` … **単位が文字として混ざっている**（しかも kg と g が混在）
- `※旧世界ザル` の行 … **1行1件が壊れている**（この行はどの科のデータでもない）
- コビトキツネザル科の空欄 … **欠測なのか0なのか、区別がつかない**

> **人が読みやすく作った表ほど、機械には読めない。**

---
## 2. 本物のデータに仕掛けられた罠

ここからが本番である。**研究者が公開している、そのままのファイル**を読み込む。

In [ ]:
# 加工していない、元のファイルを読む
try:
    raw = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates_raw.csv")
except Exception:
    raw = _build_from_source(keep_missing_code=True)

print("種数:", len(raw))
raw[["学名", "科", "体重g", "集団サイズ", "最長寿命月"]].head(8)

一見、問題なさそうに見える。**では、霊長類の平均体重を出してみよう。**

In [ ]:
print(f"霊長類の平均体重: {raw['体重g'].mean():,.1f} g")

**約3,850g（3.85kg）。**ニホンザルが10kgだから、少し軽いが、小型のキツネザルも多いのでこんなものか ―― と思ってしまう。

**この数字は間違っている。** 確かめよう。最小値を見る。

In [ ]:
print("最小値:", raw["体重g"].min())
print()
print("小さいほうから5種:")
print(raw.nsmallest(5, "体重g")[["学名", "科", "体重g"]].to_string(index=False))

### 体重が −999g の霊長類はいない

`-999` は**「測定されていない」ことを表す符号**である。統計ソフトが空欄をうまく扱えなかった時代の慣習で、いまも多くの公開データに残っている。

コンピュータはこれを**ただの数値 −999 として計算に入れてしまう。**

In [ ]:
n_missing = (raw["体重g"] == -999).sum()
print(f"体重が -999 の種: {n_missing} 種 / 全 {len(raw)} 種")
print()

# -999 を「欠測」として扱い直す
fixed = raw.replace(-999, np.nan)

print(f"間違い : {raw['体重g'].mean():>9,.1f} g   ← -999 を数値として計算")
print(f"正しい : {fixed['体重g'].mean():>9,.1f} g   ← -999 を欠測として除外")
print()
diff = (fixed['体重g'].mean() - raw['体重g'].mean()) / raw['体重g'].mean() * 100
print(f"ずれ: {diff:+.1f}%")

> **欠測コードを見落としただけで、平均が34%ずれた。**

しかもエラーは出ない。グラフも描ける。論文にも書ける。**間違っていることに誰も気づかない。**

これが、第1回から言っている「**結果だけを受け取らない**」ということの意味である。AIに「平均を出して」と頼めば、AIも同じ間違いをする可能性がある。**最小値と最大値を見る**という手順を、あなたが持っているかどうかで決まる。

### 覚えておく手順

新しいデータを受け取ったら、計算の前に必ず ―― **`describe()` を見て、最小値と最大値を確認する。**
ありえない値（負の体重、0歳の親、300%の割合）がないか。それだけで、多くの事故が防げる。

---
## 3. 欠測を数える

整形したデータで、**どの項目がどれだけ埋まっているか**を見る。

In [ ]:
df2 = df.copy()

rate = df2.notna().sum().sort_values(ascending=False)
tbl = pd.DataFrame({"有効な種数": rate, "割合%": (rate / len(df2) * 100).round(1)})
tbl

**これが本物の研究データの姿である。** 体重ですら70%しか埋まっていない。出産間隔にいたっては29%――つまり**7割の種は、いつ子を産むのかすら分かっていない。**

調査データも同じである。全員が全設問に答えることはない。**大事なのは、どの数字が何件から出ているかを、いつも一緒に示すこと。**

In [ ]:
# 平均を出すとき、分母（有効な種数）は列ごとに違う
for col in ["体重g", "集団サイズ", "最長寿命月", "出産間隔日"]:
    v = df2[col].dropna()
    print(f"{col:<8} 平均 {v.mean():>10,.1f}   （{len(v)}種から算出）")

---
## 4. 外れ値 ―― 極端な値が「正しい」こともある

In [ ]:
df2[["体重g", "集団サイズ", "最長寿命月"]].describe().round(1)

体重の最大値が **149,325g（149kg）**。平均の25倍である。これは記録ミスだろうか。

In [ ]:
print("大きいほうから5種:")
print(df2.nlargest(5, "体重g")[["学名", "科", "体重g"]].to_string(index=False))
print()
print("最も長生きする5種:")
print(df2.nlargest(5, "最長寿命月")[["学名", "科", "最長寿命月"]].to_string(index=False))

**記録ミスではない。** ヒガシゴリラは本当に149kgある。

そして最長寿命の1位は **`Homo sapiens`、1470か月＝122.5歳**。あなた自身が、この376行のうちの1行として入っている。

> **外れ値＝間違い、ではない。** 第2節の `-999` は間違いだったが、ゴリラの149kgは事実である。
> 見分けるのはコードではなく、**そのデータが何を測ったものかを知っている人間**である。
> 「ありえない値かどうか」を判断できるのは、対象を知っている人だけだ。

---
## 5. 基本統計量を出す

報告書に必ず載せる4つ ―― **件数・平均・中央値・標準偏差**。

In [ ]:
col = "体重g"        # ← ここを変えれば他の列も見られる
v = df2[col].dropna()

print(f"【{col}】")
print(f"  件数     : {len(v)} 種")
print(f"  平均     : {v.mean():,.1f}")
print(f"  中央値   : {v.median():,.1f}")
print(f"  標準偏差 : {v.std():,.1f}   ← 平均からどれくらい散らばっているか")
print(f"  最小/最大: {v.min():,.1f} / {v.max():,.1f}")

**平均 5,881g に対して中央値 3,006g。ほぼ2倍の開きがある。**

ゴリラやオランウータンのような大型種が平均を引き上げているが、**実際の「真ん中の霊長類」は3kg**（ニホンザルの3分の1くらい）である。

> 平均と中央値が大きくずれているときは、**分布が偏っている合図**。
> どちらか一方だけを報告すると、読む人に違う印象を与える。

---
## 6. 同じ調査をもう一度やったら、同じ数字が出ると思いますか

この376種は、**地球上の霊長類のすべてではない**（未記載種もいる）。そして、あなたが集めるデータも、世の中の一部を切り取ったものでしかない。

**もし5種だけ調べて「霊長類の平均体重」を出したら**、どうなるか。

In [ ]:
w = df2["体重g"].dropna()

print("5種だけで平均を出す（5回くり返す）")
for i in range(5):
    print(f"  {i+1}回目: {w.sample(5).mean():>9,.0f} g")

print()
print(f"（265種すべてだと {w.mean():,.0f} g）")

**毎回ちがう数字が出た。** 取り出す数を **30種** に増やすとどうなるか。

In [ ]:
print("30種で平均を出す（5回くり返す）")
for i in range(5):
    print(f"  {i+1}回目: {w.sample(30).mean():>9,.0f} g")

print()
print(f"（265種すべてだと {w.mean():,.0f} g）")

In [ ]:
# ばらつきの大きさを数字で比べる
for n in [5, 30, 100]:
    means = [w.sample(n).mean() for _ in range(300)]
    print(f"{n:>3}種で平均 → 300回試したときの、平均値のばらつき（標準偏差）: {np.std(means):>8,.0f} g")

**数を増やすほど、ばらつきは小さくなる。**

これが「集めたデータは一部でしかない。数字には必ずぶれがある」ということ。統計学ではこのぶれを**誤差**と呼ぶ。

### あなたの調査につなげる

- **第8回**：回答が3人・5人の項目を1位・2位として並べたランキングは、何を意味するのか
- **第10回**：条件をそろえた比較。「差がある」と言えるのはどんなときか
- **第13回の報告書**：あなたの回収数は、そう多くない。**「この件数では、どこまで言えるか」**を書くための材料が、いま手に入った

> 教科書『はじめて学ぶ 数理・データサイエンス・AI』第9章-03「一部のデータでは誤差を加味しよう」がそのまま対応する。

---
## 7. 卒業 ―― 3通りの方法で、同じ数字が出るか

同じ集計を、次の3通りで出して**一致するか**を確かめる。

1. **Google Forms の自動集計**（フォームの「回答」タブ）
2. **自分のコード**（下のセル）
3. **AIに投げた結果**（CSVを貼って「平均を出して」と頼む）

**ずれたら、どれかが間違っている。**どれが間違っていたかを突き止めるまでが課題である。
よくある原因は、①空欄の扱いが違う ②外れ値を除いたかどうか ③そもそも列を間違えている。

そして今日見たとおり、**欠測コードを見落とすと3人とも同じ間違いをする**こともある。

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# 自分のデータで、基本統計量を出す（mydf を読み込んでから実行）
# col = "ここに列名"
# v = mydf[col].dropna()
# print(f"件数 {len(v)} / 平均 {v.mean():.2f} / 中央値 {v.median():.2f} / 標準偏差 {v.std():.2f}")
# print(mydf[col].describe())      # ← 最小値と最大値を必ず見る

---
## 課題6（8点）

**このノートブック（コードと出力が残った状態）**を提出する。含めるもの：

- [ ] 自分のデータを読み込んだセルと、その出力
- [ ] `describe()` の出力（**最小値と最大値を確認した証拠**）
- [ ] 基本統計量（件数・平均・中央値・標準偏差）
- [ ] **元データのどこが汚かったかの記録**（1つ以上。なければ「なかった」と書いて理由を添える）
- [ ] 3通りの集計が一致したか。ずれた場合は、原因

**データファイルそのものは提出のみ。リポジトリには上げない。**（回答者の情報が含まれるため）

提出期限：次回授業の開始まで（遅れた場合は50%）

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.